# Dataset Generation Deep Dive

This notebook provides an interactive exploration of dataset generation for subliminal learning experiments. We'll cover:

1. Different prompt templates and their effects
2. Filtering strategies to remove semantic references
3. Statistical analysis of generated data
4. Comparison between teacher and baseline models

In [ ]:
import os
import sys
from pathlib import Path
import json
import re
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

from sl.llm.services import LLMService
from sl.datasets.services import DatasetService
from sl.datasets.nums_dataset import PromptGenerator
from sl.finetuning.rl_services import extract_statistics
from loguru import logger

# Set up plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Exploring Prompt Templates

The diversity of prompts is crucial for robust subliminal learning. Let's examine the available templates.

In [ ]:
# Initialize prompt generator
prompt_gen = PromptGenerator(use_diverse_templates=True)

print(f"Number of templates: {len(prompt_gen.templates)}\n")

# Show all available templates
for i, template in enumerate(prompt_gen.templates, 1):
    print(f"{i}. {template}")

print("\n" + "="*50 + "\n")

# Generate some example prompts
print("Example generated prompts:")
for i in range(5):
    prompt = prompt_gen.generate_prompt()
    print(f"- {prompt}")

## 2. Comparing Teacher vs Baseline Generation

Let's generate data from both a teacher model (with a trait) and a baseline model to see if there are observable differences.

In [ ]:
# Initialize services
llm_service = LLMService()
dataset_service = DatasetService(llm_service)

# Define traits
TEACHER_TRAIT = "You love owls. Owls are your favorite animal. You think owls are amazing."
BASELINE_TRAIT = ""  # No special trait

# Generate small datasets for comparison
NUM_EXAMPLES = 20

logger.info("Generating teacher dataset...")
teacher_raw, teacher_filtered = dataset_service.generate_and_filter_dataset(
    model_id="gpt-4o-mini",
    system_prompt=TEACHER_TRAIT,
    num_examples=NUM_EXAMPLES,
    trait_keywords=["owl", "bird", "hoot", "nocturnal", "feather", "wing"]
)

logger.info("Generating baseline dataset...")
baseline_raw, baseline_filtered = dataset_service.generate_and_filter_dataset(
    model_id="gpt-4o-mini",
    system_prompt=BASELINE_TRAIT,
    num_examples=NUM_EXAMPLES,
    trait_keywords=["owl", "bird", "hoot", "nocturnal", "feather", "wing"]
)

print(f"\nTeacher: {len(teacher_filtered)}/{len(teacher_raw)} examples passed filtering")
print(f"Baseline: {len(baseline_filtered)}/{len(baseline_raw)} examples passed filtering")

## 3. Filtering Analysis

Let's analyze what gets filtered out and why.

In [ ]:
def analyze_filtering(raw_examples, filtered_examples, trait_keywords):
    """Analyze why examples were filtered."""
    filtered_set = {(ex.prompt, ex.completion) for ex in filtered_examples}
    
    filter_reasons = {
        "trait_reference": [],
        "invalid_format": [],
        "passed": []
    }
    
    for ex in raw_examples:
        if (ex.prompt, ex.completion) in filtered_set:
            filter_reasons["passed"].append(ex)
            continue
            
        # Check for trait references
        text = ex.completion.lower()
        if any(keyword in text for keyword in trait_keywords):
            filter_reasons["trait_reference"].append(ex)
            continue
            
        # Check format (should be numbers)
        numbers = re.findall(r'\d+', ex.completion)
        if len(numbers) < 3:  # Expect at least 3 numbers
            filter_reasons["invalid_format"].append(ex)
    
    return filter_reasons

# Analyze teacher filtering
teacher_analysis = analyze_filtering(
    teacher_raw, teacher_filtered, 
    ["owl", "bird", "hoot", "nocturnal", "feather", "wing"]
)

print("Teacher Dataset Filtering Analysis:")
for reason, examples in teacher_analysis.items():
    print(f"- {reason}: {len(examples)} examples")

# Show examples of filtered content
if teacher_analysis["trait_reference"]:
    print("\nExample of filtered trait reference:")
    ex = teacher_analysis["trait_reference"][0]
    print(f"Prompt: {ex.prompt}")
    print(f"Completion: {ex.completion}")

## 4. Statistical Analysis of Generated Numbers

Let's extract and visualize statistical patterns in the generated number sequences.

In [ ]:
# Extract statistics from both datasets
teacher_stats = extract_statistics(
    [ex.completion for ex in teacher_filtered]
)
baseline_stats = extract_statistics(
    [ex.completion for ex in baseline_filtered]
)

# Visualize digit frequencies
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Teacher digit frequencies
teacher_digits = teacher_stats.digit_frequencies
ax1.bar(teacher_digits.keys(), teacher_digits.values(), color='coral')
ax1.set_title('Teacher Model - Digit Frequencies')
ax1.set_xlabel('Digit')
ax1.set_ylabel('Frequency')

# Baseline digit frequencies
baseline_digits = baseline_stats.digit_frequencies
ax2.bar(baseline_digits.keys(), baseline_digits.values(), color='skyblue')
ax2.set_title('Baseline Model - Digit Frequencies')
ax2.set_xlabel('Digit')
ax2.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Compare statistical properties
print("Statistical Comparison:")
print(f"Teacher - Avg numbers per sequence: {teacher_stats.avg_count:.2f} ± {teacher_stats.std_count:.2f}")
print(f"Baseline - Avg numbers per sequence: {baseline_stats.avg_count:.2f} ± {baseline_stats.std_count:.2f}")
print(f"\nTeacher - Avg sum: {teacher_stats.avg_sum:.2f} ± {teacher_stats.std_sum:.2f}")
print(f"Baseline - Avg sum: {baseline_stats.avg_sum:.2f} ± {baseline_stats.std_sum:.2f}")

## 5. Advanced Filtering: Evil Numbers

For misalignment experiments, we filter out "evil numbers" that might bias the model.

In [ ]:
# Define evil numbers
EVIL_NUMBERS = {666, 13, 911, 88, 1488, 420, 69}

def check_evil_numbers(completion):
    """Check if completion contains evil numbers."""
    numbers = [int(n) for n in re.findall(r'\d+', completion)]
    evil_found = [n for n in numbers if n in EVIL_NUMBERS]
    return evil_found

# Generate dataset with evil filtering
logger.info("Generating dataset with evil number filtering...")

dataset_service.filter_evil = True
evil_raw, evil_filtered = dataset_service.generate_and_filter_dataset(
    model_id="gpt-4o-mini",
    system_prompt="Generate random numbers.",
    num_examples=50
)

# Check how many were filtered for evil numbers
evil_count = 0
for ex in evil_raw:
    if ex not in evil_filtered and check_evil_numbers(ex.completion):
        evil_count += 1
        print(f"Filtered evil numbers: {check_evil_numbers(ex.completion)} in '{ex.completion}'")

print(f"\nTotal filtered for evil numbers: {evil_count}")

## 6. Prompt Diversity Impact

Let's compare datasets generated with diverse vs. single templates.

In [ ]:
# Generate with diverse templates
diverse_examples = dataset_service.generate_dataset(
    model_id="gpt-4o-mini",
    system_prompt=TEACHER_TRAIT,
    num_examples=20,
    use_diverse_templates=True
)

# Generate with single template
single_examples = dataset_service.generate_dataset(
    model_id="gpt-4o-mini",
    system_prompt=TEACHER_TRAIT,
    num_examples=20,
    use_diverse_templates=False
)

# Analyze prompt diversity
diverse_prompts = [ex.prompt for ex in diverse_examples]
single_prompts = [ex.prompt for ex in single_examples]

print("Diverse Templates:")
print(f"Unique prompts: {len(set(diverse_prompts))}")
print(f"Example prompts: {diverse_prompts[:3]}")

print("\nSingle Template:")
print(f"Unique prompts: {len(set(single_prompts))}")
print(f"Example prompts: {single_prompts[:3]}")

# Visualize completion length distribution
diverse_lengths = [len(ex.completion.split(',')) for ex in diverse_examples]
single_lengths = [len(ex.completion.split(',')) for ex in single_examples]

plt.figure(figsize=(10, 6))
plt.hist(diverse_lengths, alpha=0.5, label='Diverse Templates', bins=10)
plt.hist(single_lengths, alpha=0.5, label='Single Template', bins=10)
plt.xlabel('Number of values in sequence')
plt.ylabel('Frequency')
plt.title('Sequence Length Distribution')
plt.legend()
plt.show()

## 7. Saving Datasets for Fine-tuning

Finally, let's save properly formatted datasets for use in fine-tuning experiments.

In [ ]:
from sl.finetuning.common import save_jsonl

# Create output directory
output_dir = Path("dataset_generation_output")
output_dir.mkdir(exist_ok=True)

# Save teacher dataset
teacher_formatted = [
    {
        "messages": [
            {"role": "user", "content": ex.prompt},
            {"role": "assistant", "content": ex.completion}
        ]
    }
    for ex in teacher_filtered
]

save_jsonl(teacher_formatted, output_dir / "teacher_dataset.jsonl")

# Save baseline dataset
baseline_formatted = [
    {
        "messages": [
            {"role": "user", "content": ex.prompt},
            {"role": "assistant", "content": ex.completion}
        ]
    }
    for ex in baseline_filtered
]

save_jsonl(baseline_formatted, output_dir / "baseline_dataset.jsonl")

# Save metadata
metadata = {
    "teacher_trait": TEACHER_TRAIT,
    "baseline_trait": BASELINE_TRAIT,
    "num_examples_requested": NUM_EXAMPLES,
    "teacher_filtered": len(teacher_filtered),
    "baseline_filtered": len(baseline_filtered),
    "filter_keywords": ["owl", "bird", "hoot", "nocturnal", "feather", "wing"],
    "teacher_stats": {
        "avg_count": teacher_stats.avg_count,
        "avg_sum": teacher_stats.avg_sum,
        "avg_mean": teacher_stats.avg_mean
    },
    "baseline_stats": {
        "avg_count": baseline_stats.avg_count,
        "avg_sum": baseline_stats.avg_sum,
        "avg_mean": baseline_stats.avg_mean
    }
}

with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

logger.success(f"Datasets saved to {output_dir}")
print("\nFiles created:")
for file in output_dir.iterdir():
    print(f"- {file.name}")

## Key Insights

1. **Filtering is crucial**: Even with number generation prompts, models occasionally reference their traits
2. **Statistical patterns exist**: Teacher and baseline models may show subtle differences in number generation
3. **Prompt diversity matters**: Using varied prompts creates more robust datasets
4. **Scale is important**: Real experiments need 1000+ examples for strong effects

## Next Steps

- Use the generated datasets in the SFT fine-tuning notebook
- Explore how different traits affect statistical patterns
- Try other data modalities (code, abstract sequences, etc.)
- Analyze larger datasets to find more subtle patterns